In [1]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q gdown

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 227.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 132.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 29.9 MB/s eta 0:00:00


In [2]:
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch version: {torch.__version__}")

from unsloth import FastLanguageModel
print("Unsloth imported successfully")

NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB
PyTorch version: 2.10.0+cu128
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth imported successfully


In [4]:
!wget -q https://raw.githubusercontent.com/sivangichatterjee/CIS6200_FinalProject/main/logs/training_data.json

import json
with open("training_data.json") as f:
    training_data = json.load(f)

print(f"Training examples: {len(training_data)}")
print(f"\nSample instruction:\n{training_data[0]['instruction'][:300]}")
print(f"\nSample output:\n{training_data[0]['output'][:300]}")

Training examples: 957

Sample instruction:
You are a biomedical fact-checker.

CLAIM: 1 in 5 million in UK have abnormal PrP positivity.

EVIDENCE:
[1] RESULTS Of the 32,441 appendix samples 16 were positive for abnormal PrP, indicating an overall prevalence of 493 per million population (95% confidence interval 282 to 801 per million). The 

Sample output:
LABEL: CONTRADICT
REASONING: The claim that 1 in 5 million in the UK have abnormal PrP positivity is contradicted by evidence [1], which reports a prevalence of 493 per million population, significantly higher than the claimed rate. This study provides a direct estimate of abnormal PrP positivity in


In [5]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True
)
print("Base model loaded successfully")

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Base model loaded successfully


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True
)
print("LoRA applied")
model.print_trainable_parameters()

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.5.2 patched 16 layers with 16 QKV layers, 16 O layers and 0 MLP layers.


LoRA applied
trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


In [7]:
from datasets import Dataset

def format_example(example):
    text = f"""Below is an instruction. Write a response.

### Instruction:
{example['instruction']}

### Response:
{example['output']}"""
    return {"text": text}

dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_example)

print(f"Dataset ready: {len(dataset)} examples")
print(f"\nSample:\n{dataset[0]['text'][:400]}")

Map:   0%|          | 0/957 [00:00<?, ? examples/s]

Dataset ready: 957 examples

Sample:
Below is an instruction. Write a response.

### Instruction:
You are a biomedical fact-checker.

CLAIM: 1 in 5 million in UK have abnormal PrP positivity.

EVIDENCE:
[1] RESULTS Of the 32,441 appendix samples 16 were positive for abnormal PrP, indicating an overall prevalence of 493 per million population (95% confidence interval 282 to 801 per million). The prevalence in those born in 1941-60 (73


In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        bf16=True,
        fp16=False,
        logging_steps=10,
        output_dir="./student_checkpoints",
        save_strategy="epoch",
        report_to="none",
        warmup_ratio=0.1,
    )
)

print("Starting training...")
print("Expected: 15-20 minutes on this GPU\n")

trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"Final loss: {trainer_stats.training_loss:.4f}")
print(f"Time: {trainer_stats.metrics['train_runtime']/60:.1f} minutes")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/957 [00:00<?, ? examples/s]

Starting training...
Expected: 15-20 minutes on this GPU



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 957 | Num Epochs = 3 | Total steps = 180
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 3,407,872 of 1,239,222,272 (0.28% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.477620
20,2.321994
30,2.078499
40,1.954848
50,1.911374
60,1.908766
70,1.855020
80,1.861082
90,1.831396
100,1.810876


Unsloth: Restored added_tokens_decoder metadata in ./student_checkpoints/checkpoint-60/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./student_checkpoints/checkpoint-120/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./student_checkpoints/checkpoint-180/tokenizer_config.json.



Training complete!
Final loss: 1.9045
Time: 1.0 minutes


In [9]:
model.save_pretrained("./student_model_final")
tokenizer.save_pretrained("./student_model_final")
print("Saved.")

import os
print(os.listdir("./student_model_final"))

Unsloth: Restored added_tokens_decoder metadata in ./student_model_final/tokenizer_config.json.


Saved.
['tokenizer.json', 'adapter_model.safetensors', 'tokenizer_config.json', 'chat_template.jinja', 'README.md', 'adapter_config.json']


In [11]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

test_prompt = """Below is an instruction. Write a response.

### Instruction:
You are a biomedical fact-checker.

CLAIM: Aspirin reduces cardiovascular risk.

EVIDENCE:
[1] Aspirin has been shown to reduce the risk of cardiovascular events in high-risk patients by inhibiting platelet aggregation.

Analyze the claim based on the evidence above.

### Response:
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

# Add stop sequences so model doesn't ramble after CONFIDENCE line
stop_strings = ["OPINION:", "CONFIDENCEASON:", "CONFIDENCELEVEL:", "\n\n\n"]

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.1,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
    stop_strings=stop_strings,
    tokenizer=tokenizer
)
response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

# Also clean up in post-processing just in case
def clean_response(text):
    """Keep only LABEL, REASONING, CONFIDENCE lines"""
    lines = text.strip().split("\n")
    clean_lines = []
    for line in lines:
        if any(line.startswith(prefix) for prefix in ["LABEL:", "REASONING:", "CONFIDENCE:"]):
            clean_lines.append(line)
        elif clean_lines:
            # Stop at first non-standard line after we've started
            break
    return "\n".join(clean_lines)

cleaned = clean_response(response)
print("Raw response:")
print(response[:300])
print("\nCleaned response:")
print(cleaned)

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Raw response:
LABEL: SUPPORT
REASONING: The evidence supports the claim that aspirin reduces cardiovascular risk, as it is shown to inhibit platelet aggregation, which is a key factor in the development of cardiovascular events. This action of aspirin is consistent with the known cardiovascular protective effects

Cleaned response:
LABEL: SUPPORT
REASONING: The evidence supports the claim that aspirin reduces cardiovascular risk, as it is shown to inhibit platelet aggregation, which is a key factor in the development of cardiovascular events. This action of aspirin is consistent with the known cardiovascular protective effects of other antiplatelet agents, such as clopidogrel. The study by [1] specifically highlights aspirin's ability to reduce cardiovascular risk in high-risk patients, further supporting the claim.
CONFIDENCE: HIGH


In [13]:
import os

# Download index files from GitHub
!wget -q https://raw.githubusercontent.com/sivangichatterjee/CIS6200_FinalProject/main/indexes/recursive_128.faiss -O recursive_128.faiss
!wget -q https://raw.githubusercontent.com/sivangichatterjee/CIS6200_FinalProject/main/indexes/recursive_128_chunks.json -O recursive_128_chunks.json

# Download val claims
!wget -q https://raw.githubusercontent.com/sivangichatterjee/CIS6200_FinalProject/main/claims_val.json

import json
with open("claims_val.json") as f:
    val_claims = json.load(f)

print(f"Val claims: {len(val_claims)}")
print(f"FAISS size: {os.path.getsize('recursive_128.faiss') / 1e6:.1f} MB")
print(f"Chunks size: {os.path.getsize('recursive_128_chunks.json') / 1e6:.1f} MB")

Val claims: 338
FAISS size: 28.6 MB
Chunks size: 10.5 MB


In [15]:
!pip install -q faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 99.8 MB/s eta 0:00:00


In [16]:
import faiss
import numpy as np
import time
import json
from sentence_transformers import SentenceTransformer

# Load FAISS index
index = faiss.read_index("recursive_128.faiss")
with open("recursive_128_chunks.json") as f:
    chunks = json.load(f)

print(f"Index: {index.ntotal} vectors")
print(f"Chunks: {len(chunks)}")

# Load embedding model
QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("Embedding model loaded")

# Retrieval function
def retrieve_evidence(claim_text, top_k=5):
    prefixed = QUERY_INSTRUCTION + claim_text
    query_embedding = embed_model.encode(
        [prefixed], normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")
    search_k = min(top_k * 5, index.ntotal)
    scores, indices = index.search(query_embedding, search_k)
    seen_docs = {}
    for score, idx in zip(scores[0], indices[0]):
        chunk = chunks[int(idx)]
        doc_id = str(chunk["doc_id"])
        if doc_id not in seen_docs:
            seen_docs[doc_id] = {
                "doc_id": chunk["doc_id"],
                "score": float(score),
                "text": chunk["text"],
                "chunk_id": chunk["chunk_id"]
            }
        if len(seen_docs) >= top_k:
            break
    retrieved_docs = list(seen_docs.values())
    evidence_text = ""
    for i, doc in enumerate(retrieved_docs):
        evidence_text += f"[{i+1}] {doc['text']}\n\n"
    return evidence_text.strip(), retrieved_docs

# Label parser
def parse_label(response_text):
    if not response_text:
        return "PARSE_ERROR"
    for line in response_text.strip().split("\n"):
        if line.startswith("LABEL:"):
            label = line.replace("LABEL:", "").strip().upper()
            if "SUPPORT" in label:
                return "SUPPORT"
            elif "CONTRADICT" in label or "REFUTE" in label:
                return "CONTRADICT"
            elif "NOT_ENOUGH" in label:
                return "NOT_ENOUGH_INFO"
    return "PARSE_ERROR"

# Clean up repetitive output
def clean_response(text):
    lines = text.strip().split("\n")
    clean_lines = []
    for line in lines:
        if any(line.startswith(p) for p in ["LABEL:", "REASONING:", "CONFIDENCE:"]):
            clean_lines.append(line)
        elif clean_lines:
            break
    return "\n".join(clean_lines)

# Student prompt template
STUDENT_PROMPT = """Below is an instruction. Write a response.

### Instruction:
You are a biomedical fact-checker.

CLAIM: {claim}

EVIDENCE:
{evidence}

Analyze the claim based on the evidence above.

### Response:
"""

stop_strings = ["OPINION:", "CONFIDENCEASON:", "CONFIDENCELEVEL:", "\n\n\n"]

# Run inference
student_logs = []
start_time = time.time()

print(f"\nRunning student inference on {len(val_claims)} val claims...")

for i, claim in enumerate(val_claims):
    if i % 50 == 0 and i > 0:
        elapsed = (time.time() - start_time) / 60
        rate = i / max(elapsed, 0.1)
        eta = (len(val_claims) - i) / max(rate, 0.1)
        print(f"  [{i}/{len(val_claims)}] {elapsed:.1f} min elapsed, ~{eta:.0f} min remaining")

    evidence_text, retrieved_docs = retrieve_evidence(claim["claim"])

    prompt = STUDENT_PROMPT.format(
        claim=claim["claim"],
        evidence=evidence_text
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        stop_strings=stop_strings,
        tokenizer=tokenizer
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    response = clean_response(response)

    student_label = parse_label(response)

    student_logs.append({
        "claim_id":           claim["id"],
        "claim":              claim["claim"],
        "ground_truth_doc":   claim["evidence_doc_id"],
        "ground_truth_label": claim["evidence_label"],
        "retrieved_docs":     retrieved_docs,
        "evidence_text":      evidence_text,
        "student_response":   response,
        "student_label":      student_label,
        "correct":            student_label == claim["evidence_label"]
    })

    # Checkpoint every 50
    if (i + 1) % 50 == 0:
        with open("student_logs_checkpoint.json", "w") as f:
            json.dump(student_logs, f)

# Final save
with open("student_logs.json", "w") as f:
    json.dump(student_logs, f, indent=2)

total = len(student_logs)
correct = sum(1 for log in student_logs if log["correct"])
parse_err = sum(1 for log in student_logs if log["student_label"] == "PARSE_ERROR")

print(f"\nDone!")
print(f"Student accuracy: {correct/total:.1%}")
print(f"Parse errors:     {parse_err}")
print(f"Total:            {total}/338")

Index: 18638 vectors
Chunks: 18638


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

  [250/338] 10.1 min elapsed, ~4 min remaining


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [300/338] 12.1 min elapsed, ~2 min remaining


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


Done!
Student accuracy: 65.7%
Parse errors:     0
Total:            338/338


In [17]:
import json
from collections import Counter

with open("student_logs.json") as f:
    student_logs = json.load(f)

total = len(student_logs)
correct = sum(1 for log in student_logs if log["correct"])
parse_err = sum(1 for log in student_logs if log["student_label"] == "PARSE_ERROR")

print(f"Total:        {total}")
print(f"Correct:      {correct} ({correct/total:.1%})")
print(f"Wrong:        {total - correct - parse_err} ({(total-correct-parse_err)/total:.1%})")
print(f"Parse errors: {parse_err}")

print("\nStudent label distribution:")
dist = Counter(log["student_label"] for log in student_logs)
for label, count in dist.most_common():
    print(f"  {label}: {count} ({count/total:.1%})")

print("\nGround truth distribution:")
gt = Counter(log["ground_truth_label"] for log in student_logs)
for label, count in gt.most_common():
    print(f"  {label}: {count} ({count/total:.1%})")

Total:        338
Correct:      222 (65.7%)
Wrong:        116 (34.3%)
Parse errors: 0

Student label distribution:
  SUPPORT: 265 (78.4%)
  CONTRADICT: 63 (18.6%)
  NOT_ENOUGH_INFO: 10 (3.0%)

Ground truth distribution:
  SUPPORT: 216 (63.9%)
  CONTRADICT: 122 (36.1%)


In [18]:
from google.colab import files
files.download("student_logs.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>